In [7]:
!pip install PyPortfolioOpt yfinance --quiet

In [8]:
from pypfopt import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns
from pypfopt.black_litterman import BlackLittermanModel

print("PyPortfolioOpt installed successfully!")

PyPortfolioOpt installed successfully!


In [9]:
import yfinance as yf
import pandas as pd
import numpy as np

from pypfopt import EfficientFrontier, risk_models, expected_returns
from pypfopt.black_litterman import BlackLittermanModel

# Download data
tickers = ["RELIANCE.NS","TCS.NS","INFY.NS","HDFCBANK.NS","ICICIBANK.NS"]

prices = yf.download(
    tickers,
    start="2019-01-01",
    end="2025-01-01"
)["Close"]

/tmp/ipykernel_809/61134507.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices = yf.download(
[*********************100%***********************]  5 of 5 completed


In [13]:
window = 252
step = 21

markowitz_weights = []
black_weights = []

market_caps = {
    "RELIANCE.NS": 20.5e12,
    "TCS.NS": 14.8e12,
    "INFY.NS": 7.5e12,
    "HDFCBANK.NS": 13.0e12,
    "ICICIBANK.NS": 10.2e12,
}

In [15]:
for i in range(window, len(prices), step):

    data = prices.iloc[i-window:i]

    # Expected returns and covariance
    mu = expected_returns.mean_historical_return(data)
    S = risk_models.sample_cov(data)

    # =======================
    # Markowitz Portfolio
    # =======================
    ef = EfficientFrontier(mu, S)
    w = ef.min_volatility()
    markowitz_weights.append(list(w.values()))

    # =======================
    # Black-Litterman Portfolio
    # =======================

    # Investor view:
    # Reliance expected annual return = 15%
    Q = np.array([0.15])

    # Picking Reliance only
    P = np.array([
        [1, 0, 0, 0, 0]
    ])

    bl = BlackLittermanModel(
        cov_matrix=S,
        pi="market",
        market_caps=market_caps,
        Q=Q,
        P=P
    )

    bl_returns = bl.bl_returns()
    bl_cov = bl.bl_cov()

    ef_bl = EfficientFrontier(bl_returns, bl_cov)
    w_bl = ef_bl.min_volatility()

    black_weights.append(list(w_bl.values()))

In [16]:
markowitz_df = pd.DataFrame(markowitz_weights, columns=tickers)
black_df = pd.DataFrame(black_weights, columns=tickers)

markowitz_stability = markowitz_df.diff().abs().mean().mean()
black_stability = black_df.diff().abs().mean().mean()

print("="*50)
print("Portfolio Stability Comparison")
print("="*50)

print(f"Markowitz Weight Change      : {markowitz_stability:.6f}")
print(f"Black-Litterman Weight Change: {black_stability:.6f}")

print()

if black_stability < markowitz_stability:
    print("✓ Black-Litterman produced more stable portfolios.")
else:
    print("✓ Markowitz produced more stable portfolios.")

Portfolio Stability Comparison
Markowitz Weight Change      : 0.032696
Black-Litterman Weight Change: 0.024420

✓ Black-Litterman produced more stable portfolios.


In [17]:
import numpy as np

# ==================================================
# 1. Portfolio Turnover
# ==================================================

markowitz_turnover = (
    markowitz_df.diff().abs().sum(axis=1) / 2
).mean()

black_turnover = (
    black_df.diff().abs().sum(axis=1) / 2
).mean()

# ==================================================
# 2. Diversification (Herfindahl-Hirschman Index)
# Lower = More Diversified
# ==================================================

markowitz_hhi = (markowitz_df**2).sum(axis=1).mean()

black_hhi = (black_df**2).sum(axis=1).mean()

# ==================================================
# 3. Effective Number of Assets (ENA)
# Higher = Better Diversification
# ==================================================

markowitz_ena = (1 / (markowitz_df**2).sum(axis=1)).mean()

black_ena = (1 / (black_df**2).sum(axis=1)).mean()

# ==================================================
# Print Results
# ==================================================

print("="*60)
print("Portfolio Comparison")
print("="*60)

print(f"Weight Stability")
print(f"  Markowitz      : {markowitz_stability:.6f}")
print(f"  Black-Litterman: {black_stability:.6f}")

print()

print(f"Portfolio Turnover")
print(f"  Markowitz      : {markowitz_turnover:.6f}")
print(f"  Black-Litterman: {black_turnover:.6f}")

print()

print(f"Diversification (HHI)")
print(f"  Markowitz      : {markowitz_hhi:.6f}")
print(f"  Black-Litterman: {black_hhi:.6f}")

print()

print(f"Effective Number of Assets")
print(f"  Markowitz      : {markowitz_ena:.2f}")
print(f"  Black-Litterman: {black_ena:.2f}")

Portfolio Comparison
Weight Stability
  Markowitz      : 0.032696
  Black-Litterman: 0.024420

Portfolio Turnover
  Markowitz      : 0.080422
  Black-Litterman: 0.060065

Diversification (HHI)
  Markowitz      : 0.296274
  Black-Litterman: 0.293972

Effective Number of Assets
  Markowitz      : 3.60
  Black-Litterman: 3.57
